Data & System Architect
Xử lý dữ liệu, huấn luyện FP-Growth, lưu kết quả

**Output files:**
- `fp_rules_raw.parquet` — Luật thô từ FP-Growth
- `fp_association_candidates.parquet` — Top-50 gợi ý per user
- `fp_model/` — Model đã train (để reload nếu cần)
- `fp_metadata.json` — Metadata (test_start_date, total_baskets...)


In [1]:
# ============================================================
# BƯỚC 0: IMPORT & KHỞI TẠO SPARK SESSION
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.fpm import FPGrowth
import datetime

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Khởi tạo Spark với memory đủ cho FP-Growth
spark = (
    SparkSession.builder
    .appName("FPGrowth_Person1")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.driver.memory", "8g")          # Tăng nếu bị OOM
    .config("spark.executor.memory", "8g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "200") # Giảm shuffle overhead
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} khởi động thành công.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark 4.0.2 khởi động thành công.


In [3]:
# ============================================================
# BƯỚC 1: CẤU HÌNH
# ============================================================

# --- Đường dẫn ---
INPUT_FILE       = "/content/drive/MyDrive/transactions_train.csv"
DRIVE_BASE       = "/content/drive/MyDrive"

PATH_RULES       = f"{DRIVE_BASE}/fp_rules_raw.parquet"
PATH_CANDIDATES  = f"{DRIVE_BASE}/fp_association_candidates.parquet"
PATH_MODEL       = f"{DRIVE_BASE}/fp_model"
PATH_METADATA    = f"{DRIVE_BASE}/fp_metadata.json"

# Tham số thời gian
TRAIN_WEEKS      = 6           # Số tuần dùng để train

# Tham số mô hình
MAX_BASKET_SIZE  = 20          # Lọc bỏ đơn hàng quá lớn
MIN_SUPPORT      = 0.0002      # Cặp đồ phải xuất hiện ≥ 0.02% số đơn
MIN_CONFIDENCE   = 0.02        # Xác suất mua B khi đã mua A ≥ 2%

TOP_N_PREDICTIONS = 50         # Số gợi ý tối đa / khách hàng

print("Cấu hình đã sẵn sàng!")
print(f"   Input: {INPUT_FILE}")
print(f"   Train: {TRAIN_WEEKS} tuần | minSupport={MIN_SUPPORT} | minConf={MIN_CONFIDENCE}")


Cấu hình đã sẵn sàng!
   Input: /content/drive/MyDrive/transactions_train.csv
   Train: 6 tuần | minSupport=0.0002 | minConf=0.02


In [4]:
# ============================================================
# BƯỚC 2: ĐỌC DỮ LIỆU & TẠO GIỎ HÀNG (BASKET)
# ============================================================
print(f" Đang đọc dữ liệu từ: {INPUT_FILE}")

df = spark.read.option("header", "true").option("inferSchema", "true").csv(INPUT_FILE)

# Xử lý trường hợp CSV không có header
if "_c0" in df.columns:
    df = (df
          .withColumnRenamed("_c0", "t_dat")
          .withColumnRenamed("_c1", "customer_id")
          .withColumnRenamed("_c2", "article_id"))

# Ép kiểu cột ngày
df = df.withColumn("t_dat_date", F.to_date(F.col("t_dat"), "yyyy-MM-dd"))

# Tính khoảng thời gian train/test
max_date         = df.select(F.max("t_dat_date")).collect()[0][0]
test_start_date  = max_date - datetime.timedelta(days=7)
train_start_date = test_start_date - datetime.timedelta(weeks=TRAIN_WEEKS)

print(f"Khoảng train : {train_start_date}  →  {test_start_date}")
print(f"Ground truth : {test_start_date}  →  {max_date}")

# Lọc 6 tuần train
train_raw = df.filter(
    (F.col("t_dat_date") >= train_start_date) &
    (F.col("t_dat_date") <  test_start_date)
)

# Chuẩn hoá article_id: ép 10 chữ số rồi lấy 7 đầu (product_code)
train_raw = train_raw.withColumn(
    "article_id_str",
    F.substring(F.format_string("%010d", F.col("article_id").cast("int")), 1, 7)
)

# Tạo giỏ hàng: gom theo (customer, ngày)
# Dùng collect_set để tự động loại bỏ sản phẩm trùng trong cùng 1 đơn
df_baskets_full = (
    train_raw
    .groupBy("customer_id", "t_dat_date")
    .agg(F.collect_set("article_id_str").alias("items"))
)

# Lọc giỏ hàng hợp lệ
# Bỏ giỏ 1 món và giỏ quá lớn
df_baskets = df_baskets_full.filter(
    (F.size(F.col("items")) > 1) &
    (F.size(F.col("items")) <= MAX_BASKET_SIZE)
)

df_baskets.cache()
total_train_baskets = df_baskets.count()

print(f"\nSchema giỏ hàng:")
df_baskets.printSchema()
print(f"\nSố giỏ hàng hợp lệ đưa vào huấn luyện: {total_train_baskets:,}")
print(f"   minSupport={MIN_SUPPORT} → cặp đồ phải xuất hiện ≥ {int(total_train_baskets * MIN_SUPPORT):,} lần")


 Đang đọc dữ liệu từ: /content/drive/MyDrive/transactions_train.csv
Khoảng train : 2020-08-04  →  2020-09-15
Ground truth : 2020-09-15  →  2020-09-22

Schema giỏ hàng:
root
 |-- customer_id: string (nullable = true)
 |-- t_dat_date: date (nullable = true)
 |-- items: array (nullable = false)
 |    |-- element: string (containsNull = false)


Số giỏ hàng hợp lệ đưa vào huấn luyện: 320,798
   minSupport=0.0002 → cặp đồ phải xuất hiện ≥ 64 lần


In [6]:
# ============================================================
# BƯỚC 3: EDA — KHÁM PHÁ DỮ LIỆU TRAIN
# ============================================================
print("=" * 60)
print("EDA: TẬP 6 TUẦN TRAIN")
print("=" * 60)

print("\nMẫu 5 giỏ hàng:")
df_baskets.show(5, truncate=False)

print("\nThống kê kích thước giỏ hàng:")
df_baskets.select(F.size("items").alias("basket_size")) \
    .summary("count", "min", "25%", "50%", "75%", "max", "mean") \
    .show()

print("\nTop 15 sản phẩm trending nhất (6 tuần train):")
(
    df_baskets.select(F.explode("items").alias("article_id"))
    .groupBy("article_id").count()
    .withColumn("support_pct", F.round(F.col("count") / total_train_baskets * 100, 3))
    .orderBy(F.col("count").desc())
    .limit(15)
    .show(truncate=False)
)


EDA: TẬP 6 TUẦN TRAIN

Mẫu 5 giỏ hàng:
+----------------------------------------------------------------+----------+---------------------------------------------------------------+
|customer_id                                                     |t_dat_date|items                                                          |
+----------------------------------------------------------------+----------+---------------------------------------------------------------+
|005f28c032dc019b7d35e3feb6b39be48b7f73d6f8d96b265e61071813807184|2020-08-17|[0856840, 0842112]                                             |
|00880d21b352dfef30e67acc69614e15bd195525428356e55c37b1af88488963|2020-09-13|[0719957, 0869331, 0830100, 0762656, 0780297, 0903306, 0698276]|
|0099456897f76f15ffc1f5cd51db01138d8bd3fca9b63b8be8a050fe8ae5ec21|2020-08-27|[0826498, 0872511]                                             |
|00ac5e0117dd4e98de6084adebb9d46b8f07faac86088d81c769167716680dd1|2020-09-08|[0905957, 0924605, 0882902]     

In [7]:
# ============================================================
# BƯỚC 4: HUẤN LUYỆN MÔ HÌNH FP-GROWTH
# ============================================================

print(f"Bắt đầu huấn luyện FP-Growth...")
print(f"   minSupport={MIN_SUPPORT} | minConfidence={MIN_CONFIDENCE}")

fp = FPGrowth(
    itemsCol="items",
    minSupport=MIN_SUPPORT,
    minConfidence=MIN_CONFIDENCE
)

model = fp.fit(df_baskets)
print("Huấn luyện hoàn tất!")

# Lấy luật thô và kiểm tra sơ bộ
rules_raw = model.associationRules.cache()
total_rules = rules_raw.count()
print(f"   Tổng số luật sinh ra: {total_rules:,}")

if total_rules == 0:
    print("Không có luật nào! Hãy giảm MIN_SUPPORT hoặc MIN_CONFIDENCE.")
elif total_rules < 20:
    print("Quá ít luật. Cân nhắc giảm MIN_SUPPORT.")
elif total_rules > 10000:
    print("Quá nhiều luật. Cân nhắc tăng MIN_CONFIDENCE để lọc bớt.")
else:
    print("Số lượng luật hợp lý!")

# Phân phối lift nhanh
print("\nPhân phối Lift:")
rules_raw.select("lift").summary("min", "25%", "50%", "75%", "max", "mean").show()

good_rules = rules_raw.filter(F.col("lift") > 1.5).count()
print(f"   Luật có lift > 1.5 (thực sự hữu ích): {good_rules:,} / {total_rules:,}")


Bắt đầu huấn luyện FP-Growth...
   minSupport=0.0002 | minConfidence=0.02
Huấn luyện hoàn tất!
   Tổng số luật sinh ra: 1,558
Số lượng luật hợp lý!

Phân phối Lift:
+-------+------------------+
|summary|              lift|
+-------+------------------+
|    min|1.0482001918009192|
|    25%| 7.825579377364922|
|    50%|15.226775131939451|
|    75%| 38.01602982977069|
|    max| 1862.138517811705|
|   mean| 71.45663196439227|
+-------+------------------+

   Luật có lift > 1.5 (thực sự hữu ích): 1,548 / 1,558


In [8]:
# ============================================================
# BƯỚC 5: KIỂM TRA NHANH — TOP 10 LUẬT TỐT NHẤT
# ============================================================
print("Top 10 luật theo Lift:")
(
    rules_raw
    .orderBy(F.col("lift").desc())
    .select(
        "antecedent",
        "consequent",
        F.round("confidence", 4).alias("confidence"),
        F.round("lift",       4).alias("lift"),
        F.round("support",    4).alias("support"),
    )
    .show(10, truncate=False)
)


Top 10 luật theo Lift:
+----------+----------+----------+---------+-------+
|antecedent|consequent|confidence|lift     |support|
+----------+----------+----------+---------+-------+
|[0909827] |[0909823] |0.5573    |1862.1385|2.0E-4 |
|[0909823] |[0909827] |0.7604    |1862.1385|2.0E-4 |
|[0725663] |[0725662] |0.6628    |1497.3375|4.0E-4 |
|[0725662] |[0725663] |0.8028    |1497.3375|4.0E-4 |
|[0841565] |[0913540] |0.4826    |1268.9587|3.0E-4 |
|[0913540] |[0841565] |0.7951    |1268.9587|3.0E-4 |
|[0917300] |[0917297] |0.4779    |1216.8458|2.0E-4 |
|[0917297] |[0917300] |0.5159    |1216.8458|2.0E-4 |
|[0933889] |[0903590] |0.4146    |1064.1104|2.0E-4 |
|[0903590] |[0933889] |0.544     |1064.1104|2.0E-4 |
+----------+----------+----------+---------+-------+
only showing top 10 rows


In [9]:
# ============================================================
# BƯỚC 6: SINH GỢI Ý CÁ NHÂN HOÁ (TOP-50 / USER)
# ============================================================
print("1.Tổng hợp lịch sử mua sắm của từng khách hàng...")

# Gom toàn bộ lịch sử 6 tuần của mỗi user thành 1 set
user_history = (
    train_raw
    .groupBy("customer_id")
    .agg(F.collect_set("article_id_str").alias("items"))
)

print("2.Transform: quét lịch sử và sinh gợi ý qua association rules...")
# model.transform() tự khớp lịch sử với các luật đã học,
# trả về cột 'prediction' chứa danh sách sản phẩm gợi ý (sắp xếp theo confidence)
predictions = model.transform(user_history)

print(f"3.Lọc user có gợi ý và cắt Top {TOP_N_PREDICTIONS}...")
association_candidates = (
    predictions
    .filter(F.size(F.col("prediction")) > 0)
    .withColumn(
        "fp_candidates",
        F.expr(f"slice(prediction, 1, {TOP_N_PREDICTIONS})")
    )
    .select("customer_id", "fp_candidates")
)

total_users = association_candidates.count()
print(f"\nSinh gợi ý cho {total_users:,} khách hàng.")
association_candidates.show(5, truncate=False)


1.Tổng hợp lịch sử mua sắm của từng khách hàng...
2.Transform: quét lịch sử và sinh gợi ý qua association rules...
3.Lọc user có gợi ý và cắt Top 50...

Sinh gợi ý cho 206,373 khách hàng.
+----------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|customer_id                                                     |fp_candidates                                                                                                                                                                                                                                                                                                                          

In [10]:
# =============================
# BƯỚC 7: LƯU TẤT CẢ KẾT QUẢ
# =============================
import json as _json

print("Đang lưu các file output...")

# 1. Lưu luật thô
print(f"   Lưu rules_raw → {PATH_RULES}")
rules_raw.write.mode("overwrite").parquet(PATH_RULES)

# 2. Lưu danh sách gợi ý per-user
print(f"   Lưu association_candidates → {PATH_CANDIDATES}")
association_candidates.write.mode("overwrite").parquet(PATH_CANDIDATES)

# 3. Lưu model để tái sử dụng
print(f"   Lưu model → {PATH_MODEL}")
model.save(PATH_MODEL)

# 4. Lưu metadata
metadata = {
    "test_start_date":      str(test_start_date),
    "train_start_date":     str(train_start_date),
    "max_date":             str(max_date),
    "total_train_baskets":  total_train_baskets,
    "total_rules":          total_rules,
    "total_candidate_users": total_users,
    "MIN_SUPPORT":           MIN_SUPPORT,
    "MIN_CONFIDENCE":        MIN_CONFIDENCE,
    "TRAIN_WEEKS":           TRAIN_WEEKS,
    "TOP_N_PREDICTIONS":     TOP_N_PREDICTIONS,
    "INPUT_FILE":            INPUT_FILE,
}
with open(PATH_METADATA, "w") as f:
    _json.dump(metadata, f, indent=2, default=str)
print(f"   Lưu metadata → {PATH_METADATA}")


Đang lưu các file output...
   Lưu rules_raw → /content/drive/MyDrive/fp_rules_raw.parquet
   Lưu association_candidates → /content/drive/MyDrive/fp_association_candidates.parquet
   Lưu model → /content/drive/MyDrive/fp_model
   Lưu metadata → /content/drive/MyDrive/fp_metadata.json
